In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import torch
from torchvision.utils import save_image
from diffusion import create_diffusion
from diffusers.models import AutoencoderKL
from models import DiT_models
import argparse


def parse_args(args=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", type=str, choices=list(DiT_models.keys()), default="DiT-XL/2")
    parser.add_argument("--vae", type=str, choices=["ema", "mse"], default="mse")
    parser.add_argument("--image-size", type=int, choices=[128, 256, 512], default=256)
    parser.add_argument("--num-classes", type=int, default=1000)
    parser.add_argument("--cfg-scale", type=float, default=4.0)
    parser.add_argument("--num-sampling-steps", type=int, default=250)
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--ckpt", type=str, default=None,
                        help="Optional path to a DiT checkpoint (default: auto-download a pre-trained DiT-XL/2 model).")
    parser.add_argument("--lora-ckpt", type=str, default=None)
    parser.add_argument("--lora-ckpt-cl", type=str, default=None)
    parser.add_argument("--convnext", type=int, default=0)
    parser.add_argument("--lora_scale", type=int, default=0)
    return parser.parse_args(args)

sim_args = [
    "--model", "DiT-B/2",
    "--vae", "mse",
    "--image-size", "128",
    "--num-classes", "30",
    "--cfg-scale", "4.0",
    "--num-sampling-steps", "100",
    "--seed", "0",
    "--convnext", "1",
    
    "--ckpt", "/home/jichang/workspace/model/canvas10_ulcl/add_lora/t1_0140000.pt",
    # "--lora-ckpt", "/home/jichang/workspace/model/canvas10_ulcl/add_lora/t2_lora_0300000.pt",
    # "--lora-ckpt-cl", "/home/jichang/workspace/model/canvas10_ulcl/add_lora/t3_lora_1100000.pt",
]
args = parse_args(sim_args)

/home/jichang/workspace/.dm_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Setup PyTorch:
torch.manual_seed(args.seed)
torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"

# Load model:
latent_size = args.image_size // 8
# latent_size = args.image_size

model = DiT_models[args.model](
    input_size=latent_size,
    num_classes=args.num_classes,
    in_channels=4,
    y_channels=4,
    convnext=True if args.convnext == 1 else False
).to(device)


with_lora = 0
scale_lora = args.lora_scale
if args.lora_ckpt is not None or args.lora_ckpt_cl is not None:
    checkpoint = torch.load(args.ckpt)
    model.load_state_dict(checkpoint["model"], strict=False)
    from tools.lora import update_model, update_model_scale
    
    # Initialize LoRA structure
    if scale_lora == 1:
        update_model_scale(model, rank=4, alpha=8, device='cuda')
    else:
        update_model(model, rank=4, alpha=8, device='cuda')
    
    # Case 1: Both lora_ckpt and lora_ckpt_cl are provided
    if args.lora_ckpt is not None and args.lora_ckpt_cl is not None:
        # Load first LoRA checkpoint and merge into base weights
        lora_checkpoint1 = torch.load(args.lora_ckpt, map_location='cuda')
        lora_params1 = lora_checkpoint1["lora_params"]
        lora_params1 = {name.replace('module.', ''): param for name, param in lora_params1.items()}
        
        with torch.no_grad():
            for name, param in model.named_parameters():
                if name.endswith('pretrained.weight') and name.replace('pretrained.weight', 'A.weight') in lora_params1:
                    # Merge first LoRA into base weights: W = W + B*A * (alpha/rank)
                    A = lora_params1[name.replace('pretrained.weight', 'A.weight')]
                    B = lora_params1[name.replace('pretrained.weight', 'B.weight')]
                    scaling = lora_checkpoint1.get('alpha', 8) / lora_checkpoint1.get('rank', 4)
                    param.add_(B @ A * scaling)
        
        # Load second LoRA checkpoint as trainable parameters
        lora_checkpoint2 = torch.load(args.lora_ckpt_cl)
        lora_params2 = lora_checkpoint2["lora_params"]
        lora_params2 = {name.replace('module.', ''): param for name, param in lora_params2.items()}
        
        for name, param in model.named_parameters():
            if name in lora_params2:
                param.data.copy_(lora_params2[name])
        
        print(f"Merged first LoRA into base weights and loaded second trainable LoRA")
    
    # Case 2: Only lora_ckpt is provided
    elif args.lora_ckpt is not None:
        lora_checkpoint = torch.load(args.lora_ckpt)
        lora_params = lora_checkpoint["lora_params"]
        lora_params = {name.replace('module.', ''): param for name, param in lora_params.items()}
        
        for name, param in model.named_parameters():
            if name in lora_params:
                param.data.copy_(lora_params[name])
    
    # Case 3: Only lora_ckpt_cl is provided
    elif args.lora_ckpt_cl is not None:
        lora_checkpoint = torch.load(args.lora_ckpt_cl)
        lora_params = lora_checkpoint["lora_params"]
        lora_params = {name.replace('module.', ''): param for name, param in lora_params.items()}
        
        for name, param in model.named_parameters():
            if name in lora_params:
                param.data.copy_(lora_params[name])

elif with_lora == True:
    from tools.lora import update_model
    if scale_lora == 1:
        update_model_scale(model, rank=4, alpha=8, device='cuda')
    else:
        update_model(model, rank=4, alpha=8, device='cuda')
    checkpoint = torch.load(args.ckpt)
    model.load_state_dict(checkpoint["model"])
else:
    checkpoint = torch.load(args.ckpt)
    model.load_state_dict(checkpoint["model"])

model.eval()
diffusion = create_diffusion(str(args.num_sampling_steps))

vae = AutoencoderKL.from_pretrained(f"stabilityai/sd-vae-ft-{args.vae}").to(device)
# load_path = "local_vae/"
# vae = AutoencoderKL.from_pretrained(load_path).to(device)

## Inference

In [ ]:
import torch
import random
import numpy as np
from PIL import Image as PILImage
from IPython.display import display

# 固定随机种子
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# model_inference = model_noised
model_inference = model

# Labels to condition the model with:
nested_list = [[k for _ in range(1)] for k in range(18)]
class_labels = [element for sublist in nested_list for element in sublist]

# Create sampling noise:
n = len(class_labels)
z = torch.randn(n, 4, latent_size, latent_size, device=device)
y = torch.tensor(class_labels, device=device)

# Setup classifier-free guidance:
z = torch.cat([z, z], 0)
y_null = torch.tensor([args.num_classes] * n, device=device)
y = torch.cat([y, y_null], 0)
model_kwargs = dict(y=y, cfg_scale=args.cfg_scale)

# Sample images:
samples = diffusion.p_sample_loop(
    model_inference.forward_with_cfg, z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=True, device=device
)
samples, _ = samples.chunk(2, dim=0)  # Remove null class samples
samples = vae.decode(samples / 0.18215).sample

# Split samples into three groups (0-5, 6-11, 12-17)
group1 = samples[:6]  # classes 0-5
group2 = samples[6:12]  # classes 6-11
group3 = samples[12:]  # classes 12-17

# Save and display each group separately
for i, group in enumerate([group1, group2, group3]):
    image_path = f"images_output/test_group_{i+1}.png"
    save_image(group, image_path, nrow=6, normalize=True, value_range=(-1, 1))  # 修改为nrow=6
    
    # Show the image using PIL
    print(f"Group {i+1} (Classes {i*6}-{i*6+5}):")
    pil_img = PILImage.open(image_path)
    display(pil_img)